# 05 — Average precision is a numerical integral

**Work package:** WP8 · **Runtime:** CPU, < 1 min · runs from committed result files (no dataset needed)
**Learning objectives:** numerical integration (17), integration error and convergence (18),
Richardson extrapolation / Romberg (19), Gaussian quadrature (20), shape-preserving interpolation (8).

Average precision for one class is the area under its precision–recall curve,

$$\mathrm{AP} = \int_0^1 \hat p(r)\,dr, \qquad \hat p(r) = \max_{k:\, r_k \ge r} p_k ,$$

where $\hat p$ (the *precision envelope*) is a **non-increasing step function** with jumps at the
recall values reached by the ranked detections. Because it is piecewise constant, the integral can
be computed **exactly** (sum of rectangle areas). Libraries instead use sampled rules
(COCO: 101 points; VOC2007: 11 points; Ultralytics: trapezoid on a linearly interpolated envelope).

Questions:
1. How large are the differences between rules on *our* PR curves?
2. Do textbook convergence rates (trapezoid $O(h^2)$, Simpson $O(h^4)$) hold for this integrand?
3. Does Richardson extrapolation (Romberg) help?
4. Does the quadrature error matter compared with the **sampling** uncertainty of mAP?

Data: YOLO11n (blocked split) predictions on blocked VALID, matched with our tested
`porkqc.metrics` (confidence-ordered matching, IoU 0.5).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from porkqc import CLASS_NAMES
from porkqc.plotting import apply_style, SERIES, MUTED
from porkqc.predictions import load_json
from porkqc.metrics.evaluate import match_dataset, class_pr_inputs, summarise, bootstrap, ImageMatches
from porkqc.metrics.ap import pr_curve, precision_envelope, interpolated_precision, AP_METHODS, ap_exact
from porkqc.metrics.quadrature import midpoint_rule, trapezoid_rule, simpson_rule, gauss_legendre_rule, romberg_table, observed_order
from porkqc.data.audit import frame_number
apply_style()
FIG = ROOT / "results" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
OUT = ROOT / "results" / "ap_study"; OUT.mkdir(parents=True, exist_ok=True)

gt = load_json(ROOT / "results/ground_truth/blocked_valid.json")
pred = load_json(ROOT / "results/predictions/xeval_yolo11n_640_blocked_on_blocked_valid_default.json")
matches = match_dataset(gt, pred, len(CLASS_NAMES))
for m in matches:
    m.group = str(frame_number(m.file) // 50)
curves = {}
for c in (3, 1):   # unsealed, packaging-error (headline classes, D-024)
    tp, sc, n_gt = class_pr_inputs(matches, c)
    r, p = pr_curve(tp[:, 0], sc, n_gt)
    curves[CLASS_NAMES[c]] = (r, p, n_gt, len(sc))
    print(f"{CLASS_NAMES[c]:16s} ground-truth boxes {n_gt:4d}, predictions {len(sc):5d}, max recall {r[-1]:.3f}")

unsealed         ground-truth boxes   59, predictions   623, max recall 0.983
packaging-error  ground-truth boxes  117, predictions  1926, max recall 0.983


## 1. The integrand: raw precision–recall points and the step envelope

The raw curve zig-zags (each false positive lowers precision, each true positive raises it).
The envelope takes the running maximum from the right: it is the **monotone, shape-preserving**
interpolant of the points — it never overshoots, unlike a polynomial or spline through them
(compare PCHIP vs cubic spline, LO 8). Linear interpolation between envelope points (Ultralytics)
slightly *adds* area where the step drops.

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (name, (r, p, n_gt, n_pred)) in zip(axes, curves.items()):
    rr = np.linspace(0, 1, 2001)
    ax.plot(r, p, color=SERIES[0], lw=1, alpha=0.6, label="raw precision")
    ax.step(np.concatenate(([0], r)), np.concatenate(([precision_envelope(p)[0]], precision_envelope(p))),
            where="pre", color=SERIES[1], label="envelope (exact integrand)")
    ax.fill_between(rr, interpolated_precision(r, p, rr), step=None, color=SERIES[1], alpha=0.12)
    ax.axvline(r[-1], color=MUTED, ls=":", lw=1)
    ax.set_title(f"{name}: {n_gt} boxes, exact AP50 = {ap_exact(r, p):.3f}")
    ax.set_xlabel("recall"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
axes[0].set_ylabel("precision"); axes[0].legend(loc="lower left")
fig.tight_layout(); fig.savefig(FIG / "05_pr_envelope.png", dpi=150); plt.show()

## 2. Library rules vs the exact integral (per class and mAP)

In [3]:
rows = []
for name, (r, p, *_ ) in curves.items():
    ex = ap_exact(r, p)
    rows.append({"class": name, **{m: AP_METHODS[m](r, p) for m in AP_METHODS},
                 **{f"{m} - exact": AP_METHODS[m](r, p) - ex for m in AP_METHODS if m != "exact"}})
per_class = pd.DataFrame(rows).set_index("class")
display(per_class.round(4))
mp = pd.DataFrame([{ "rule": m, "mAP50 (classes with GT)": summarise(matches, CLASS_NAMES, m)["map50"],
                     "mAP50-95": summarise(matches, CLASS_NAMES, m)["map50_95"]} for m in AP_METHODS]).set_index("rule")
display(mp.round(4))

,exact,coco101,voc11,ultralytics,coco101 - exact,voc11 - exact,ultralytics - exact
class,,,,,,,
unsealed,0.8070,0.8053,0.7722,0.8084,-0.0016,-0.0347,0.0014
packaging-error,0.6866,0.6847,0.6634,0.6865,-0.0019,-0.0232,-0.0000


,mAP50 (classes with GT),mAP50-95
rule,,
exact,0.6091,0.3350
coco101,0.6080,0.3350
voc11,0.5918,0.3337
ultralytics,0.6084,0.3346


## 3. Convergence: generic quadrature rules applied to the envelope

Treat $\hat p(r)$ as a black-box integrand on $[0,1]$ and integrate it with $N$ panels/nodes,
comparing with the exact area. For a **smooth** integrand the error falls like $h^2$ (trapezoid,
midpoint) or $h^4$ (Simpson) and Gauss–Legendre converges spectrally. A step function has jumps,
so the panel containing each jump contributes an $O(h)$ error that no polynomial rule can remove.
A smooth control integrand ($e^r$) is run through exactly the same code to show the rules are
implemented correctly (the textbook orders appear there).

In [4]:
Ns = np.array([8, 16, 32, 64, 128, 256, 512, 1024, 2048])
def study(f, exact):
    res = {"midpoint": [], "trapezoid": [], "Simpson": [], "Gauss-Legendre": []}
    for n in Ns:
        res["midpoint"].append(abs(midpoint_rule(f, 0, 1, n) - exact))
        res["trapezoid"].append(abs(trapezoid_rule(f, 0, 1, n) - exact))
        res["Simpson"].append(abs(simpson_rule(f, 0, 1, n) - exact))
        res["Gauss-Legendre"].append(abs(gauss_legendre_rule(f, 0, 1, min(n, 200)) - exact))
    return {k: np.array(v) for k, v in res.items()}

r, p, *_ = curves["unsealed"]
f_step = lambda x: interpolated_precision(r, p, x)
err_step = study(f_step, ap_exact(r, p))
err_smooth = study(np.exp, np.e - 1)
h = 1.0 / Ns
orders = pd.DataFrame({
    "smooth e^r": {k: observed_order(h[:5], v[:5]) for k, v in err_smooth.items() if k != "Gauss-Legendre"},
    "AP envelope (unsealed)": {k: observed_order(h, v) for k, v in err_step.items() if k != "Gauss-Legendre"},
}).round(2)
print("Observed order p in |error| ~ h^p (log-log least-squares slope):"); display(orders)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, (title, err) in zip(axes, [("smooth control: e^r", err_smooth), ("AP envelope (unsealed, IoU 0.5)", err_step)]):
    for k, col in zip(err, SERIES):
        e = np.where(err[k] > 0, err[k], np.nan)
        ax.loglog(Ns, e, marker="o", color=col, label=k)
    ax.loglog(Ns, 0.5 / Ns, color=MUTED, ls="--", lw=1, label="slope -1 (reference)")
    ax.set_title(title); ax.set_xlabel("number of panels / nodes N"); ax.set_ylabel("|error|")
axes[1].legend(loc="lower left")
fig.tight_layout(); fig.savefig(FIG / "05_quadrature_convergence.png", dpi=150); plt.show()

Observed order p in |error| ~ h^p (log-log least-squares slope):


,smooth e^r,AP envelope (unsealed)
midpoint,2.0,0.97
trapezoid,2.0,0.79
Simpson,4.0,0.99


## 4. Richardson extrapolation (Romberg) on a step function

Romberg assumes the trapezoid error expands as $c_1 h^2 + c_2 h^4 + \dots$ and cancels terms by
combining step sizes. With jumps the leading error is $O(h)$ and erratic, so the extrapolated
columns need not improve — and can get worse.

In [5]:
levels = 10
Rs = romberg_table(f_step, 0, 1, levels); Rm = romberg_table(np.exp, 0, 1, levels)
exact_s, exact_m = ap_exact(r, p), np.e - 1
rom = pd.DataFrame({
    "panels": 2 ** np.arange(levels),
    "step: trapezoid err": np.abs(Rs[:, 0] - exact_s),
    "step: Romberg diag err": np.abs(np.diag(Rs) - exact_s),
    "smooth: trapezoid err": np.abs(Rm[:, 0] - exact_m),
    "smooth: Romberg diag err": np.abs(np.diag(Rm) - exact_m),
})
display(rom.style.format({c: "{:.2e}" for c in rom.columns if "err" in c}))
ratio = rom["step: trapezoid err"].values[:-1] / rom["step: trapezoid err"].values[1:]
print("error ratio between successive halvings (trapezoid, step):", np.round(ratio, 2), "- Romberg assumes 4")

,panels,step: trapezoid err,step: Romberg diag err,smooth: trapezoid err,smooth: Romberg diag err
0,1,3.07e-01,3.07e-01,1.41e-01,1.41e-01
1,2,9.64e-02,2.63e-02,3.56e-02,5.79e-04
2,4,8.99e-03,2.33e-02,8.94e-03,8.59e-07
3,8,2.49e-03,5.10e-03,2.24e-03,3.35e-10
4,16,3.36e-03,3.44e-03,5.59e-04,3.33e-14
5,32,8.45e-04,2.77e-03,1.40e-04,8.88e-16
6,64,8.97e-04,7.86e-04,3.50e-05,1.11e-15
7,128,5.88e-04,4.49e-04,8.74e-06,6.66e-16
8,256,9.17e-05,1.23e-04,2.18e-06,4.44e-16
9,512,1.71e-04,2.75e-04,5.46e-07,2.22e-16


error ratio between successive halvings (trapezoid, step): [ 3.18 10.73  3.61  0.74  3.98  0.94  1.52  6.41  0.53] - Romberg assumes 4


## 5. Does any of this matter? Quadrature error vs sampling uncertainty

The real question for an engineer: is the choice of integration rule large compared with how
uncertain mAP is anyway because the validation set is small? Sampling uncertainty: bootstrap over
50-frame runs (independent units, V12).

In [6]:
stat = lambda mm: summarise(mm, CLASS_NAMES)["map50"]
bs = bootstrap(matches, stat, n_boot=1000, seed=0, grouped=True)
spread = mp["mAP50 (classes with GT)"].max() - mp["mAP50 (classes with GT)"].min()
summary = {"mAP50_exact": float(mp.loc["exact", "mAP50 (classes with GT)"]),
           "rule_spread_mAP50": float(spread),
           "bootstrap_ci_width_mAP50": bs["high"] - bs["low"], "bootstrap_std_mAP50": bs["std"],
           "ratio_ci_width_to_rule_spread": (bs["high"] - bs["low"]) / spread if spread else float("inf"),
           "observed_orders": orders.to_dict()}
print(json.dumps(summary, indent=1))
json.dump(summary, open(OUT / "summary.json", "w"), indent=1)
per_class.to_csv(OUT / "per_class_rules.csv"); mp.to_csv(OUT / "map_by_rule.csv"); rom.to_csv(OUT / "romberg.csv", index=False)

{
 "mAP50_exact": 0.6091193698373598,
 "rule_spread_mAP50": 0.017357388474326996,
 "bootstrap_ci_width_mAP50": 0.30172827061335816,
 "bootstrap_std_mAP50": 0.05813409249628769,
 "ratio_ci_width_to_rule_spread": 17.383275776747123,
 "observed_orders": {
  "smooth e^r": {
   "midpoint": 2.0,
   "trapezoid": 2.0,
   "Simpson": 4.0
  },
  "AP envelope (unsealed)": {
   "midpoint": 0.97,
   "trapezoid": 0.79,
   "Simpson": 0.99
  }
 }
}


## 6. Conclusions (filled in from the outputs above)

See `docs/RESULTS_LOG.md` R5 for the written interpretation used in the report.